# Spesenabrechnungsanalyse

Dieses Notebook zeigt, wie man Agenten erstellt, die Plugins verwenden, um Reisekosten aus lokalen Belegbildern zu verarbeiten, eine Spesenabrechnung per E-Mail zu generieren und Ausgabendaten mit einem Tortendiagramm zu visualisieren. Agenten wählen dynamisch Funktionen basierend auf dem Aufgabenkontext aus.

Schritte:
1. OCR-Agent verarbeitet das lokale Belegbild und extrahiert Reisekostendaten.
2. E-Mail-Agent generiert eine Spesenabrechnung per E-Mail.

### Beispiel für ein Reisekostenszenario:
Stellen Sie sich vor, Sie sind ein Mitarbeiter, der für ein Geschäftstreffen in einer anderen Stadt reist. Ihr Unternehmen hat eine Richtlinie, alle angemessenen reiserelevanten Ausgaben zu erstatten. Hier ist eine Aufschlüsselung der potenziellen Reisekosten:
- Transport:
Flugticket für eine Hin- und Rückreise von Ihrer Heimatstadt zur Zielstadt.
Taxi- oder Mitfahrdienste zum und vom Flughafen.
Lokaler Transport in der Zielstadt (wie öffentliche Verkehrsmittel, Mietwagen oder Taxis).

- Unterkunft:
Hotelaufenthalt für drei Nächte in einem Mittelklasse-Business-Hotel in der Nähe des Tagungsorts.

- Verpflegung:
Tägliche Verpflegungspauschale für Frühstück, Mittag- und Abendessen, basierend auf der Richtlinie des Unternehmens.

- Sonstige Ausgaben:
Parkgebühren am Flughafen.
Internetgebühren im Hotel.
Trinkgelder oder kleine Servicegebühren.

- Dokumentation:
Sie reichen alle Belege (Flüge, Taxis, Hotel, Verpflegung usw.) und einen ausgefüllten Spesenbericht zur Erstattung ein.


## Erforderliche Bibliotheken importieren

Importieren Sie die notwendigen Bibliotheken und Module für das Notebook.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Definieren von Ausgabenmodellen

Erstellen Sie ein Pydantic-Modell für einzelne Ausgaben und eine ExpenseFormatter-Klasse, um eine Benutzeranfrage in strukturierte Ausgabedaten umzuwandeln.

Jede Ausgabe wird im Format dargestellt:
`{'date': '07-Mar-2025', 'description': 'Flug zum Zielort', 'amount': 675.99, 'category': 'Transport'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Definieren von Tools – E-Mail generieren

Erstellen Sie eine Tool-Funktion zum Generieren einer E-Mail für die Einreichung eines Spesenanspruchs.
- Dieses Tool verwendet den `@tool` Dekorator aus dem Microsoft Agent Framework.
- Es berechnet den Gesamtbetrag der Ausgaben und formatiert die Details in den Textkörper der E-Mail.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Werkzeug zum Extrahieren von Reisekosten aus Belegbildern

Erstellen Sie eine Werkzeugfunktion, um Reisekosten aus Belegbildern zu extrahieren.
- Dieses Werkzeug verwendet den `@tool` Dekorator aus dem Microsoft Agent Framework.
- Es liest das Belegbild ein, kodiert es als base64 und gibt die Daten-URI zurück, damit der Agent es analysieren kann.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Verarbeitung von Ausgaben

Definieren Sie die Agenten und verbinden Sie sie in einem sequenziellen Workflow mit `WorkflowBuilder`.
- Der OCR-Agent extrahiert strukturierte Ausgabendaten aus dem Belegbild mithilfe des Tools `load_receipt_image`.
- Der E-Mail-Agent nimmt die extrahierten Daten und erstellt eine professionelle Spesenerstattungs-E-Mail mit dem Tool `generate_expense_email`.
- `WorkflowBuilder` erstellt mit `add_edge` eine sequenzielle Pipeline: OCR-Agent → E-Mail-Agent.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Hauptfunktion

Erstellen Sie den sequentiellen Arbeitsablauf und führen Sie ihn aus, um das Belegbild zu verarbeiten und die Spesenabrechnung per E-Mail zu erstellen.


> **Hinweis:** Dieser Workflow übergibt derzeit das Kassenbonbild als Base64-Text, welchen die meisten Chatmodelle (einschließlich gpt-4o) nicht als Bild behandeln.
> Es kann auch das Kontextfenster des Modells überschreiten. Es ist besser, die OCR mit Azure AI Vision (oder einem anderen OCR-Tool) durchzuführen und nur den extrahierten Text zu übergeben oder den Ablauf so zu ändern, dass das Bild als `image_url` Nachricht gesendet wird.
> Wenn Sie nur Kontextfehler vermeiden möchten, versuchen Sie ein kleineres Kassenbonbild oder ein Modell mit einem größeren Kontextfenster.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
